<a href="https://www.kaggle.com/code/mimakhdumiiitm/22f3001418-notebook-26t1?scriptVersionId=305758348" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
print("done")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/sample_submission.csv
/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv
/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv


KeyboardInterrupt: 

In [11]:
# ── DEBUG CELL: Understand your data ─────────────────────────
import os
import pandas as pd

DATA_DIR = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1'

print("="*70)
print("📁 EXPLORING KAGGLE INPUT DIRECTORY")
print("="*70)

# Find all folders and count files (without printing each file)
for dirname, subdirs, filenames in os.walk(DATA_DIR):
    # Calculate depth for indentation
    depth = dirname.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * depth
    
    # Print folder name
    print(f"{indent}📂 {os.path.basename(dirname)}/")
    
    # Print subdirectories
    for subdir in subdirs:
        print(f"{indent}  📁 {subdir}/")
    
    # Count and show files (don't print all 17k files!)
    if filenames:
        # Show first 3 files as sample
        sample_files = filenames[:3]
        print(f"{indent}  📄 Files: {len(filenames)} total")
        print(f"{indent}     Sample: {sample_files}")
    
    # Only go 2 levels deep
    if depth >= 2:
        break

print("\n" + "="*70)
print("📊 CHECKING CSV FILES")
print("="*70)

# Find all CSV files
csv_files = []
for dirname, _, filenames in os.walk(DATA_DIR):
    for f in filenames:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(dirname, f))

for csv_path in csv_files:
    print(f"\n📄 {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   First 2 IDs: {df.iloc[:2, 0].tolist() if len(df) > 0 else 'Empty'}")

📁 EXPLORING KAGGLE INPUT DIRECTORY
📂 26-t-1-dl-gen-ainppe-1/
  📁 images/
  📄 Files: 3 total
     Sample: ['sample_submission.csv', 'train.csv', 'test.csv']
  📂 images/
    📄 Files: 88337 total
       Sample: ['32258b566bba41018fbfbc8cce95d6dc.png', 'c653d938386447e58ab9a53f7100fa46.png', '1760d71394de469aa5fb39efbeee905b.png']

📊 CHECKING CSV FILES

📄 /kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/sample_submission.csv
   Shape: (10, 21)
   Columns: ['id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'Pneumoperitoneum', 'Pneumomediastinum', 'Subcutaneous Emphysema', 'Tortuous Aorta', 'Calcification of the Aorta', 'No Finding']
   First 2 IDs: ['7b647fbfcc874a7084a4470fc150e267.png', 'cc804b94d80c4a80a206298c307adfec.png']

📄 /kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv
   Shape: (51043, 21)
   Columns: ['id', 'Atelectasis', 'C

In [12]:
# Method 01 - socre : -5.93
# ============================================================
# THORACIC PATHOLOGY DETECTION FROM CHEST X-RAY IMAGES
# Multi-class Classification | 20 Classes | Kaggle Competition
# ============================================================

# ── CELL 1: Install & Imports ────────────────────────────────
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── CELL 2: Configuration ────────────────────────────────────
class CFG:
    # ============================================================
    # PATHS - Based on your actual structure
    # ============================================================
    COMPETITION_NAME = '26-t-1-dl-gen-ainppe-1'
    DATA_DIR = f'/kaggle/input/competitions/{COMPETITION_NAME}'
    
    # All images in ONE folder (train + test together)
    IMAGE_DIR = os.path.join(DATA_DIR, 'images')
    
    # CSV files
    TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')           # ← 17,015 IDs
    SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')
    
    # Classes
    CLASSES = [
        'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
        'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
        'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia',
        'Pneumothorax', 'Pneumoperitoneum', 'Pneumomediastinum',
        'Subcutaneous Emphysema', 'Tortuous Aorta',
        'Calcification of the Aorta', 'No Finding'
    ]
    NUM_CLASSES = 20
    
    # Training
    IMG_SIZE = 224
    BATCH_SIZE = 32
    EPOCHS = 20
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    SEED = 42
    MODEL_NAME = 'efficientnet_b3'
    USE_AMP = True
    NUM_WORKERS = 2
    FN_PENALTY = 5
    FP_PENALTY = 1

print("✅ Configuration loaded")
print(f"   IMAGE_DIR: {CFG.IMAGE_DIR}")
print(f"   Total images in folder: {len(os.listdir(CFG.IMAGE_DIR))}")

# ── CELL 3: Reproducibility ──────────────────────────────────
def set_seed(seed=CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── CELL 4: Custom Metric (Competition Score) ────────────────
def competition_score(y_true, y_pred, num_classes=20):
    """
    Score_c = (TP_c - FP_c - 5*FN_c) / N_c
    Final  = mean over all classes
    """
    scores = []
    for c in range(num_classes):
        true_c = (y_true == c)
        pred_c = (y_pred == c)

        TP = np.sum(true_c & pred_c)
        FP = np.sum(~true_c & pred_c)
        FN = np.sum(true_c & ~pred_c)
        N  = np.sum(true_c)

        if N == 0:
            continue   # skip classes with no samples

        score_c = (TP - FP - CFG.FN_PENALTY * FN) / N
        scores.append(score_c)

    return np.mean(scores)

# ── CELL 5: Data Loading ─────────────────────────────────────
train_df = pd.read_csv(CFG.TRAIN_CSV)
test_df = pd.read_csv(CFG.TEST_CSV)        # ← Load ALL 17,015 test IDs
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

print(f"Train CSV    : {train_df.shape}")
print(f"Test CSV     : {test_df.shape}")   # Should be (17015, 1)
print(f"Sample sub   : {sample_sub.shape}") # Will be (10, 21) - ignore this

print(f"\n✅ Test data loaded: {len(test_df)} samples")
print(f"   First 3 test IDs: {test_df['id'].head(3).tolist()}")

# Verify test IDs exist in images folder
sample_test_ids = test_df['id'].head(3).tolist()
for img_id in sample_test_ids:
    img_path = os.path.join(CFG.IMAGE_DIR, img_id)
    exists = os.path.exists(img_path)
    print(f"   {img_id}: {'✅ Found' if exists else '❌ Missing'}")

train_df.head(3)

# ── CELL 6: Label Engineering ────────────────────────────────
# Convert one-hot → single integer label
train_df['label'] = train_df[CFG.CLASSES].values.argmax(axis=1)

print("Class distribution:")
print(train_df['label'].value_counts().sort_index()
      .rename(index=dict(enumerate(CFG.CLASSES))))

# ── CELL 7: Train / Validation Split ────────────────────────
train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label'],
    random_state=CFG.SEED
)
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# ── CELL 8: Dataset Class ────────────────────────────────────
class XRayDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df        = df
        self.image_dir = image_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['id'])

        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            # Fallback: blank image if file missing
            image = Image.fromarray(np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3),
                                             dtype=np.uint8))

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row['id']

        label = int(row['label'])
        return image, label

# ── CELL 9: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── CELL 10: Weighted Sampler (handle class imbalance) ───────
def get_weighted_sampler(df):
    labels       = df['label'].values
    class_counts = np.bincount(labels, minlength=CFG.NUM_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler

# ── CELL 11: DataLoaders ─────────────────────────────────────
train_dataset = XRayDataset(train_data, CFG.IMAGE_DIR, train_transform)  # ← Same IMAGE_DIR
val_dataset   = XRayDataset(val_data,   CFG.IMAGE_DIR, val_transform)    # ← Same IMAGE_DIR

sampler = get_weighted_sampler(train_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    sampler=sampler,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── CELL 12: Focal Loss ──────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss to handle class imbalance.
    Combines class weights + focusing parameter gamma.
    """
    def __init__(self, class_weights=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.class_weights = class_weights
        self.gamma         = gamma
        self.reduction     = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets,
                                  weight=self.class_weights,
                                  reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal.mean()
        return focal.sum()

# ── CELL 13: Compute Class Weights ───────────────────────────
labels_arr    = train_data['label'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(CFG.NUM_CLASSES),
    y=labels_arr
)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Class weights (first 5):", class_weights[:5].round(3))

# ── CELL 14: Model Definition ────────────────────────────────
def build_model(model_name='efficientnet_b3', num_classes=20, pretrained=True):
    """Build a pretrained backbone with custom classification head."""

    if 'efficientnet' in model_name:
        if model_name == 'efficientnet_b3':
            model = models.efficientnet_b3(pretrained=pretrained)
        elif model_name == 'efficientnet_b4':
            model = models.efficientnet_b4(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    elif model_name == 'densenet121':
        model = models.densenet121(pretrained=pretrained)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model

model = build_model(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {CFG.MODEL_NAME} | Trainable params: {total_params:,}")

# ── CELL 15: Loss, Optimizer, Scheduler ──────────────────────
criterion = FocalLoss(class_weights=class_weights_tensor, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY
)

scheduler = CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)
scaler    = GradScaler(enabled=CFG.USE_AMP)

# ── CELL 16: Training Function ───────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast(enabled=CFG.USE_AMP):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(loader)} "
                  f"| Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    score    = competition_score(np.array(all_labels), np.array(all_preds))
    return avg_loss, score

# ── CELL 17: Validation Function ─────────────────────────────
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(enabled=CFG.USE_AMP):
                logits = model(images)
                loss   = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    score    = competition_score(np.array(all_labels), np.array(all_preds))
    return avg_loss, score

# ── CELL 18: Training Loop ───────────────────────────────────
best_val_score = -np.inf
history = {'train_loss': [], 'val_loss': [],
           'train_score': [], 'val_score': []}

for epoch in range(1, CFG.EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{CFG.EPOCHS}")
    print(f"{'='*60}")

    train_loss, train_score = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch
    )
    val_loss, val_score = validate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_score'].append(train_score)
    history['val_score'].append(val_score)

    print(f"\nTrain Loss: {train_loss:.4f} | Train Score: {train_score:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Score: {val_score:.4f}")
    print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

    # Save best model
    if val_score > best_val_score:
        best_val_score = val_score
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✅ Best model saved! Score: {best_val_score:.4f}")

print(f"\n🏆 Best Validation Score: {best_val_score:.4f}")

# ── CELL 19: Plot Training History ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='red')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_score'], label='Train Score', color='blue')
axes[1].plot(history['val_score'],   label='Val Score',   color='red')
axes[1].set_title('Competition Score Curve')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

# ── CELL 20: Test Dataset & Prediction ───────────────────────
class XRayTestDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df        = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id   = self.df.iloc[idx]['id']
        img_path = os.path.join(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.fromarray(
                np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
            )
        if self.transform:
            image = self.transform(image)
        return image, img_id

# TTA (Test-Time Augmentation) transforms
tta_transforms = [
    val_transform,  # Original
    transforms.Compose([   # Horizontal flip
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ]),
    transforms.Compose([   # Slight rotation
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.RandomRotation(degrees=(5, 5)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ]),
]

def predict_with_tta(model, df, image_dir, tta_list):
    """Run inference with TTA and average softmax probabilities."""
    model.eval()
    all_probs = None

    for t_idx, transform in enumerate(tta_list):
        print(f"  TTA pass {t_idx+1}/{len(tta_list)} ...")
        test_dataset = XRayTestDataset(df, image_dir, transform)
        test_loader  = DataLoader(
            test_dataset,
            batch_size=CFG.BATCH_SIZE,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )

        probs_list = []
        img_ids    = []

        with torch.no_grad():
            for images, ids in test_loader:
                images = images.to(device, non_blocking=True)
                with autocast(enabled=CFG.USE_AMP):
                    logits = model(images)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                probs_list.append(probs)
                img_ids.extend(ids)

        probs_arr = np.concatenate(probs_list, axis=0)

        if all_probs is None:
            all_probs = probs_arr
        else:
            all_probs += probs_arr

    all_probs /= len(tta_list)
    return np.array(img_ids), all_probs

# ── CELL 21: Load Best Model & Predict ───────────────────────
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pth', map_location=device))
print("✅ Best model loaded for inference.")

# ============================================================
# CRITICAL FIX: Use test_df (17,015 rows) NOT sample_sub (10 rows)
# ============================================================
print(f"\n🔮 Running predictions on {len(test_df)} test images...")
print(f"   Using image directory: {CFG.IMAGE_DIR}")

# Verify we have the right dataframe
assert len(test_df) == 17015, f"❌ Expected 17015 test samples, got {len(test_df)}"
print(f"✅ Confirmed: {len(test_df)} test samples ready")

# Run TTA predictions
img_ids, avg_probs = predict_with_tta(
    model, 
    test_df,          # ← Use test_df (17,015 IDs), NOT sample_sub (10 IDs)!
    CFG.IMAGE_DIR,    # ← Same folder for train and test
    tta_transforms
)

# Final predicted class
predicted_classes = avg_probs.argmax(axis=1)

print(f"\n📊 Prediction Results:")
print(f"   Total predictions: {len(predicted_classes)}")
print(f"   Expected: 17015")
assert len(predicted_classes) == 17015, "❌ Prediction count mismatch!"
print(f"✅ Prediction count correct!")
print(f"   Unique classes predicted: {np.unique(predicted_classes)}")

# ── CELL 22: Generate Submission CSV ─────────────────────────
def create_submission(img_ids, predicted_classes, classes):
    """
    Create one-hot encoded submission.
    Each row: one '1' in the predicted class column, rest '0'.
    """
    submission = pd.DataFrame({'id': img_ids})

    # Initialize all class columns to 0
    for cls in classes:
        submission[cls] = 0

    # Set predicted class to 1
    for idx, pred_class in enumerate(predicted_classes):
        class_name = classes[pred_class]
        submission.at[idx, class_name] = 1

    # Ensure correct column order (id first, then classes in original order)
    column_order = ['id'] + classes
    submission = submission[column_order]

    # Convert to int
    for cls in classes:
        submission[cls] = submission[cls].astype(int)

    return submission

submission = create_submission(img_ids, predicted_classes, CFG.CLASSES)

# ============================================================
# VALIDATION
# ============================================================
print(f"\n📋 Submission Validation:")
print(f"   Total rows: {len(submission)}")
print(f"   Expected: 17015")

# Check shape
assert submission.shape == (17015, 21), \
    f"❌ Wrong shape! Expected (17015, 21), got {submission.shape}"
print(f"✅ Shape correct: {submission.shape}")

# Check each row has exactly one '1'
row_sums = submission[CFG.CLASSES].sum(axis=1)
assert (row_sums == 1).all(), \
    f"❌ Some rows don't have exactly one '1'! Unique sums: {row_sums.unique()}"
print(f"✅ Each row has exactly one '1'")

# Check columns match sample_submission
sample_cols = pd.read_csv(CFG.SAMPLE_SUB).columns.tolist()
assert submission.columns.tolist() == sample_cols, \
    "❌ Column order doesn't match sample_submission!"
print(f"✅ Column order matches sample_submission")

# Preview
print(f"\n📄 Submission Preview:")
print(submission.head())
print(f"\n{submission.tail()}")

# Save
submission.to_csv('submission.csv', index=False)
print("\n✅ submission.csv saved!")
print(f"   File size: {os.path.getsize('submission.csv'):,} bytes")

# ── CELL 23: Submission Statistics ───────────────────────────
pred_dist = submission[CFG.CLASSES].idxmax(axis=1).value_counts()
print("\n📊 Prediction Distribution:")
for cls, cnt in pred_dist.items():
    pct = cnt / len(submission) * 100
    print(f"  {cls:<30}: {cnt:>5} ({pct:.2f}%)")

# ── CELL 24: Per-class Confidence Analysis ───────────────────
print("\n📊 Average Confidence per Predicted Class:")
for i, cls in enumerate(CFG.CLASSES):
    mask = (predicted_classes == i)
    if mask.sum() > 0:
        avg_conf = avg_probs[mask, i].mean()
        print(f"  {cls:<35}: {avg_conf:.4f} (n={mask.sum()})")

Using device: cuda
✅ Configuration loaded
   IMAGE_DIR: /kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images
   Total images in folder: 88337
Train CSV    : (51043, 21)
Test CSV     : (17015, 1)
Sample sub   : (10, 21)

✅ Test data loaded: 17015 samples
   First 3 test IDs: ['7b647fbfcc874a7084a4470fc150e267.png', 'cc804b94d80c4a80a206298c307adfec.png', '1df09c3becd04de995244caae36ddf57.png']
   7b647fbfcc874a7084a4470fc150e267.png: ✅ Found
   cc804b94d80c4a80a206298c307adfec.png: ✅ Found
   1df09c3becd04de995244caae36ddf57.png: ✅ Found
Class distribution:
label
Atelectasis                    2351
Cardiomegaly                    600
Consolidation                   651
Edema                           326
Effusion                       2156
Emphysema                       172
Fibrosis                        389
Hernia                           37
Infiltration                   5206
Mass                           1249
Nodule                         1527
Pleural_Thickening             

100%|██████████| 47.2M/47.2M [00:00<00:00, 132MB/s] 


Model: efficientnet_b3 | Trainable params: 11,493,436

EPOCH 1/20
  Epoch 1 | Batch 100/1356 | Loss: 37.1522
  Epoch 1 | Batch 200/1356 | Loss: 25.2253
  Epoch 1 | Batch 300/1356 | Loss: 17.4568
  Epoch 1 | Batch 400/1356 | Loss: 16.0306
  Epoch 1 | Batch 500/1356 | Loss: 12.6155
  Epoch 1 | Batch 600/1356 | Loss: 12.7757
  Epoch 1 | Batch 700/1356 | Loss: 8.9651
  Epoch 1 | Batch 800/1356 | Loss: 9.9187
  Epoch 1 | Batch 900/1356 | Loss: 6.5324
  Epoch 1 | Batch 1000/1356 | Loss: 6.4961
  Epoch 1 | Batch 1100/1356 | Loss: 8.6673
  Epoch 1 | Batch 1200/1356 | Loss: 5.3967
  Epoch 1 | Batch 1300/1356 | Loss: 5.6241

Train Loss: 18.5599 | Train Score: -3.9405
Val   Loss: 3.3852 | Val   Score: -18.1418
LR: 0.000099
✅ Best model saved! Score: -18.1418

EPOCH 2/20


KeyboardInterrupt: 